# Eksplorasi Dataset dan Persiapan Teks Semantik

## Tujuan dan Ruang Lingkup

Notebook ini mendokumentasikan Phase 2–3: audit dataset, pembentukan master dataset tanpa ID duplikat, penyusunan `semantic_text`, dan pemeriksaan panjang token sebelum encoding SBERT.

Fokusnya adalah pipeline fitur teks semantik. Fitur metadata, cosine similarity formal, fusion alpha, recommendation engine, dan metrik evaluasi **belum** termasuk di tahap ini.


## Load Dataset Mentah dan Audit Ringkasan

- Import `load_raw_movies`, `validate_movies`, dan `DATA_RAW_DIR`.
- Menentukan lokasi CSV dari `DATA_RAW_DIR`.
- Membaca CSV menjadi DataFrame `movies`.
- Membuat laporan audit `report`.
- Mencetak jumlah baris, kolom, ID unik, grup ID duplikat, dan ID kosong.


In [1]:
from movie_recommender.data.loader import load_raw_movies
from movie_recommender.data.validator import validate_movies
from movie_recommender.utils.paths import DATA_RAW_DIR

RAW_PATH = DATA_RAW_DIR / "TMDB  IMDB Movies Dataset.csv"

movies = load_raw_movies(RAW_PATH)
report = validate_movies(movies)

print("rows:", report["n_rows"])
print("columns:", report["n_columns"])
print("unique ids:", report["n_unique_ids"])
print("duplicate id groups:", report["n_duplicate_ids"])
print("null ids:", report["n_null_ids"])

rows: 443610
columns: 29
unique ids: 429755
duplicate id groups: 3463
null ids: 0


Output yang tersimpan menunjukkan dataset mentah punya 443.610 baris dan 29 kolom. Cell ini juga menjadi sumber variabel movies dan report untuk cell-cell setelahnya.

## Tabel Missing dan Empty Seluruh Kolom

Ini menampilkan tabel kolom yang punya nilai kosong. Tabelnya membedakan:
- n_missing: nilai NaN.
- n_empty: string kosong atau whitespace.
- n_blank: total keduanya.
- pct_blank: persentase kosong dari jumlah baris.

In [2]:
report["missing"]

,n_missing,n_empty,n_blank,pct_blank
release_date,25344,0,25344,5.71
backdrop_path,256596,0,256596,57.84
homepage,387845,3,387848,87.43
overview,44973,64,45037,10.15
poster_path,79352,0,79352,17.89
tagline,349602,0,349602,78.81
genres,83793,0,83793,18.89
production_companies,180228,0,180228,40.63
production_countries,121839,0,121839,27.47
spoken_languages,109749,0,109749,24.74


Contoh yang kelihatan di output: overview punya 44.973 missing dan 64 whitespace-only.

## Audit Kolom yang Relevan untuk penelitian

Cell ini membuat tabel dari report["relevant_stats"], lalu .T membalik tabelnya agar nama kolom, seperti overview, tagline, dan keywords, jadi baris.

Tujuannya fokus melihat kondisi tujuh kolom yang dipakai dalam rancangan fitur teks dan metadata, bukan seluruh 29 kolom.

In [3]:
import pandas as pd

pd.DataFrame(report["relevant_stats"]).T

,n_missing,n_empty
overview,44973,64
tagline,349602,0
keywords,271970,0
genres,83793,0
directors,10003,0
writers,64952,0
cast,69641,0


## Contoh Overview yang Hanya Berisi Whitespace

Cell ini memilih baris yang:
overview-nya bukan NaN, tetapi setelah spasi depan-belakang dibuang, isinya menjadi string kosong.

Lalu cell menampilkan contoh film beserta ID, judul, overview, tagline, dan keywords. Ini membantu melihat langsung bentuk data yang terhitung kosong meskipun bukan NaN.

In [4]:
blank_overview = movies[
    movies["overview"].notna()
    & movies["overview"].astype(str).str.strip().eq("")
]

blank_overview[["id", "title", "overview", "tagline", "keywords"]].head()

,id,title,overview,tagline,keywords
146479,64798,Garrison,,NaN,NaN
240639,1169535,We Were Fine,,NaN,NaN
249240,1157723,Anna,,NaN,NaN
269927,1373367,Notes on Blindness: Rainfall,,NaN,NaN
270959,1378522,C.K.O.D.2 - Evacuation Plan,,NaN,NaN


## Contoh baris dengan ID Duplikat

Cell ini mengambil semua baris yang ID-nya muncul lebih dari sekali, kemudian menampilkan beberapa contoh.

Tujuannya memeriksa duplikat secara visual, bukan menghapusnya. Contohnya memperlihatkan bahwa baris dengan ID sama bisa memiliki data cast yang berbeda.

In [5]:
duplicate_ids = movies[movies["id"].duplicated(keep=False)]

duplicate_ids.sort_values("id").head(10)[
    ["id", "title", "overview", "genres", "cast"]
]

,id,title,overview,genres,cast
14238,2979,Return of the Ghostbusters,The mile high city is rocked when an ancient E...,"Comedy, Fantasy, Science Fiction",NaN
14239,2979,Return of the Ghostbusters,The mile high city is rocked when an ancient E...,"Comedy, Fantasy, Science Fiction","Tim Johnson, J. Michael Weiss, Joel Hanson, Ro..."
22370,5804,The Key to Reserva,Finding an unfinished script written by Alfred...,"Thriller, Comedy","Simon Baker, Kelli O'Hara, Michael Stuhlbarg, ..."
22369,5804,The Key to Reserva,Finding an unfinished script written by Alfred...,"Thriller, Comedy","Simon Baker, Kelli O'Hara, Michael Stuhlbarg, ..."
22372,5804,The Key to Reserva,Finding an unfinished script written by Alfred...,"Thriller, Comedy","Simon Baker, Kelli O'Hara, Michael Stuhlbarg, ..."
22371,5804,The Key to Reserva,Finding an unfinished script written by Alfred...,"Thriller, Comedy",NaN
172203,6080,Death on the Nile: Making of Featurette,Featurette documenting the making of the film ...,Documentary,NaN
172202,6080,Death on the Nile: Making of Featurette,Featurette documenting the making of the film ...,Documentary,"Jane Birkin, Lois Chiles, Bette Davis, Mia Far..."
20666,6962,The Hunt for the Hidden Relic,"STEFFEN VOGT, aged around 25, is helping out a...","Adventure, Science Fiction, Fantasy, Thriller","Matthias Koeberlin, Naike Rivelli, Manou Lubow..."
20667,6962,The Hunt for the Hidden Relic,"STEFFEN VOGT, aged around 25, is helping out a...","Adventure, Science Fiction, Fantasy, Thriller","Matthias Koeberlin, Manou Lubowski, Naike Rive..."


## Deduplicate dan Mengecek Jumlah Data Film Setelah Diproses

Aturan deduplikasi yang dipakai: untuk setiap `id`, pertahankan baris dengan jumlah kolom relevan terisi terbanyak. Jika seri, pilih teks terpanjang setelah whitespace dibersihkan; jika masih seri, pilih baris yang lebih awal.

Ketujuh kolom relevan diberi bobot sama. Ini **heuristik pemilihan record**, bukan ukuran kualitas film. Implementasinya ada di `deduplicate_movies()`.


In [6]:
from movie_recommender.data.deduplicate import deduplicate_movies

movies_dedup = deduplicate_movies(movies)

print("before:", movies.shape)
print("after:", movies_dedup.shape)
print("duplicate IDs:", movies_dedup["id"].duplicated().sum())

before: (443610, 29)
after: (429755, 29)
duplicate IDs: 0


## Menyimpan master dataset ke Parquet

Cell ini membuat folder tujuan jika belum ada, lalu menyimpan movies_dedup ke Parquet.

Parquet adalah format untuk data hasil olahan. Cell ini menyimpan data, belum memverifikasi apakah hasil bacanya sama dengan data sebelum disimpan.

In [7]:
from movie_recommender.utils.paths import MOVIES_PARQUET

MOVIES_PARQUET.parent.mkdir(parents=True, exist_ok=True)

movies_dedup.to_parquet(
    MOVIES_PARQUET,
    index=False,
    compression="zstd",
)

### Verifikasi Parquet Setelah Dibaca Ulang

File Parquet dibaca kembali lalu dibandingkan dengan DataFrame sebelum disimpan. `assert_frame_equal` memeriksa isi, nama kolom, urutan, dan tipe data. Kalau tidak ada error, proses simpan-baca dianggap konsisten.


In [8]:
reloaded = pd.read_parquet(MOVIES_PARQUET)

pd.testing.assert_frame_equal(reloaded, movies_dedup)

## Menghitung Ketersediaan Teks dan Metadata

Cell ini membentuk dua mask boolean:

- has_semantic_text: apakah setidaknya satu dari overview, tagline, atau keywords berisi teks.
- has_any_metadata: apakah setidaknya satu dari genres, directors, writers, atau cast berisi teks.

Lalu mencetak jumlah film unik, yang punya teks semantik, yang tidak punya teks semantik, dan ketersediaan metadata pada kelompok itu.

Ini hanya menghitung ketersediaan, belum membuat fitur metadata atau semantic_text.

In [9]:
text_columns = ["overview", "tagline", "keywords"]
metadata_columns = ["genres", "directors", "writers", "cast"]

has_semantic_text = (
    movies_dedup[text_columns]
    .fillna("")
    .astype(str)
    .apply(lambda series: series.str.strip().ne(""))
    .any(axis=1)
)

has_any_metadata = (
    movies_dedup[metadata_columns]
    .fillna("")
    .astype(str)
    .apply(lambda series: series.str.strip().ne(""))
    .any(axis=1)
)

print("unique movies:", len(movies_dedup))
print("semantic eligible:", int(has_semantic_text.sum()))
print("no semantic text:", int((~has_semantic_text).sum()))
print(
    "no semantic text but has metadata:",
    int(((~has_semantic_text) & has_any_metadata).sum()),
)
print(
    "no semantic text and no metadata:",
    int(((~has_semantic_text) & (~has_any_metadata)).sum()),
)

unique movies: 429755
semantic eligible: 396254
no semantic text: 33501
no semantic text but has metadata: 33405
no semantic text and no metadata: 96


## Membuat Teks Semantik

`build_semantic_text()` menggabungkan `overview`, `tagline`, dan `keywords` menjadi satu teks berlabel. Field yang kosong dilewati; jika ketiganya kosong, hasilnya string kosong dan film tetap berada di master dataset.

Punctuation, stopwords, dan negasi tidak dihapus. Nilai tagline yang persis `none`, `n/a`, atau `-` diperlakukan sebagai placeholder kosong. Asumsi ini dicatat sebagai bagian dari preprocessing.


In [10]:
from movie_recommender.preprocessing.text import build_semantic_text

semantic_text = build_semantic_text(movies_dedup)

print("same number of rows:", len(semantic_text) == len(movies_dedup))
print("same index:", semantic_text.index.equals(movies_dedup.index))
print("empty semantic texts:", int(semantic_text.eq("").sum()))

same number of rows: True
same index: True
empty semantic texts: 33501


### Beberapa contoh yang memiliki teks

Contoh ini dipakai untuk memeriksa apakah label `Overview:`, `Tagline:`, dan `Keywords:` muncul sesuai field yang tersedia. Teks lengkap tidak ditampilkan agar tabel tetap terbaca.


In [11]:
example = movies_dedup.loc[
    movies_dedup["title"].isin(["Inception", "Interstellar"]),
    ["id", "title"],
].copy()

example["semantic_text"] = semantic_text.loc[example.index]
example

,id,title,semantic_text
0,27205,Inception,"Overview: Cobb, a skilled thief who commits co..."
1,157336,Interstellar,Overview: The adventures of a group of explore...


### Cek beberapa baris yang hasilnya kosong

Baris dengan `semantic_text` kosong seharusnya berasal dari film yang ketiga kolom sumbernya juga kosong. Pemeriksaan ini memastikan tidak ada teks yang hilang karena kesalahan penggabungan.


In [12]:
empty_examples = movies_dedup.loc[
    semantic_text.eq(""),
    ["id", "title", "overview", "tagline", "keywords"],
].head(5)

empty_examples

,id,title,overview,tagline,keywords
10072,919952,Il grande giorno,NaN,NaN,NaN
10867,17413,Incognito,NaN,NaN,NaN
11050,48805,L'allenatore nel pallone 2,NaN,NaN,NaN
11177,77000,Bar Sport,NaN,NaN,NaN
11398,57114,"Amore, bugie e calcetto",NaN,NaN,NaN


## Load model dan cek batas sequence length

Checkpoint yang dipakai adalah SBERT dengan backbone DistilBERT. Model memetakan teks ke ruang vektor 768 dimensi dan menghasilkan embedding yang sudah ternormalisasi.

`max_seq_length` dibaca langsung dari konfigurasi model, bukan diasumsikan. Nilai ini menjadi batas jumlah token per teks saat encoding.


In [13]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/multi-qa-distilbert-cos-v1"

model = SentenceTransformer(MODEL_NAME, device="cuda")
tokenizer = model[0].tokenizer
max_seq_length = model.max_seq_length

print("model:", MODEL_NAME)
print("max sequence length:", max_seq_length)
print("tokenizer:", tokenizer.name_or_path)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

model: sentence-transformers/multi-qa-distilbert-cos-v1
max sequence length: 512
tokenizer: sentence-transformers/multi-qa-distilbert-cos-v1


## Memilih teks yang bukan NaN

Hanya teks yang memiliki karakter non-whitespace yang diteruskan ke tokenisasi dan encoding. Film dengan `semantic_text` kosong tetap disimpan di master dataset, tetapi tidak punya representasi semantik.


In [14]:
eligible_mask = semantic_text.notna() & semantic_text.str.strip().ne("")
texts = semantic_text.loc[eligible_mask]

print("texts to inspect:", len(texts))

texts to inspect: 396254


## Hitung token tanpa truncation

Panjang diukur menggunakan tokenizer model, bukan jumlah kata. Satu kata dapat dipecah menjadi beberapa subword, dan token khusus ikut dihitung.

`truncation=False` sengaja dipakai agar yang terukur adalah panjang penuh. Peringatan tokenizer bahwa sequence melebihi 512 bukan error—itu informasi bahwa teks tersebut akan terpotong saat encoding.


In [15]:
TOKEN_BATCH_SIZE = 2048
token_lengths = []

for start in range(0, len(texts), TOKEN_BATCH_SIZE):
    batch = texts.iloc[start : start + TOKEN_BATCH_SIZE].tolist()

    encoded = tokenizer(
        batch,
        add_special_tokens=True,
        truncation=False,
        padding=False,
        return_length=True,
    )

    token_lengths.extend(encoded["length"])

token_lengths = pd.Series(
    token_lengths,
    index=texts.index,
    name="token_length",
)

print("measured texts:", len(token_lengths))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (558 > 512). Running this sequence through the model will result in indexing errors


measured texts: 396254


## Ringkas distribusi panjang

Distribusi membantu melihat apakah mayoritas teks aman atau banyak yang mendekati batas. Persentil ke-90, 95, dan 99 lebih informatif daripada nilai maksimum, karena maksimum mudah dipengaruhi satu record ekstrem.


In [16]:
token_lengths.describe(percentiles=[0.50, 0.90, 0.95, 0.99])

count    396254.000000
mean         72.914252
std          46.976944
min           5.000000
50%          61.000000
90%         140.000000
95%         168.000000
99%         215.000000
max         782.000000
Name: token_length, dtype: float64

### Proporsi teks yang melewati batas model

Batas keras model adalah `max_seq_length`. Teks yang lebih panjang akan dipotong saat encoding, sehingga bagian setelah batas tidak ikut membentuk embedding.

Karena urutan teks adalah overview, tagline, lalu keywords, bagian akhir berisiko terpotong lebih dulu.


In [17]:
n_over_limit = int(token_lengths.gt(max_seq_length).sum())
pct_over_limit = (
    n_over_limit / len(token_lengths) * 100
    if len(token_lengths)
    else 0.0
)

print("max sequence length:", max_seq_length)
print("texts exceeding limit:", n_over_limit)
print("percentage exceeding limit:", f"{pct_over_limit:.4f}%")

max sequence length: 512
texts exceeding limit: 16
percentage exceeding limit: 0.0040%


### Contoh teks yang melewati batas

Tabel di bawah menunjukkan film dengan teks terpanjang beserta jumlah tokennya. Cuplikan teks membantu memeriksa bagian mana yang berpotensi terpotong.


In [18]:
over_limit_lengths = token_lengths[token_lengths > max_seq_length]
long_indices = over_limit_lengths.sort_values(ascending=False).head(10).index

long_examples = movies_dedup.loc[
    long_indices,
    ["id", "title"],
].copy()

long_examples["token_length"] = token_lengths.loc[long_indices]
long_examples["semantic_text_preview"] = (
    semantic_text.loc[long_indices].str.slice(0, 500)
)

long_examples

,id,title,token_length,semantic_text_preview
211503,77994,Yegor's Grief,782,Overview: У Егора проблемы. Он занял денег для...
213506,64060,Sokrovishche,760,Overview: Это – новогодняя комедия-сказка. Нев...
211238,77831,This better not to know,745,"Overview: Говорить о том, что ревность - это ч..."
217837,70139,Kogda My Byli Schastlivy,735,Overview: В жизни бизнесмена Глеба все было ид...
181498,64380,Nezhnost K Revushemu Zveryu,691,Overview: По мотивам одноименного романа А. Ба...
199542,73054,Super-Mother-in-Law For Loser,660,Overview: Леонид - злоумышленник средней руки....
321167,64184,Volga River Flows,648,Overview: В заброшенный поселок на берегу Волг...
218182,92735,Padeniye Kondora,647,Overview: Действие происходит в вымышленном го...
218976,70945,Dzisay,621,"Overview: ""Дзисай"" в переводе с древнеяпонског..."
94234,71047,First Encounter - Last Encounter,607,Overview: История о событиях в Петербурге в ка...


### Perbandingan dengan panjang saat training

Model card menyebut model dilatih pada teks hingga sekitar 250 word piece. Angka ini bukan batas keras, tetapi teks yang jauh lebih panjang berada di luar rentang yang paling banyak dilihat model saat training.

Perbandingan ini dicatat sebagai keterbatasan, bukan alasan mengubah batas model atau memotong input dengan cara berbeda.


In [19]:
training_reference_length = 250

n_over_training_reference = int(
    token_lengths.gt(training_reference_length).sum()
)
pct_over_training_reference = (
    n_over_training_reference / len(token_lengths) * 100
)

print("texts over 250 tokens:", n_over_training_reference)
print(
    "percentage over 250 tokens:",
    f"{pct_over_training_reference:.2f}%",
)

texts over 250 tokens: 520
percentage over 250 tokens: 0.13%


### Benchmark awal: `model.encode` langsung

Sebelum memakai fungsi reusable, encoding 500 teks diuji langsung dengan `model.encode`. Tujuannya memastikan model berjalan di CUDA dan mengukur kecepatan kasar. Angka waktu ini menjadi pembanding untuk implementasi fungsi pada bagian berikutnya.


In [20]:
import time

import numpy as np
import torch

print("CUDA available:", torch.cuda.is_available())
print("model device:", model.device)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA belum tersedia di PyTorch")

# Sampel acak yang bisa diulang dengan seed yang sama.
benchmark_texts = texts.sample(n=500, random_state=42)

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
start_time = time.perf_counter()

benchmark_embeddings = model.encode(
    benchmark_texts.tolist(),
    batch_size=8,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

torch.cuda.synchronize()
elapsed_seconds = time.perf_counter() - start_time

benchmark_ids = movies_dedup.loc[benchmark_texts.index, "id"].to_numpy()
embedding_norms = np.linalg.norm(benchmark_embeddings, axis=1)

print("sample size:", len(benchmark_texts))
print("embedding shape:", benchmark_embeddings.shape)
print("ID count:", len(benchmark_ids))
print("elapsed seconds:", round(elapsed_seconds, 2))
print("texts per second:", round(len(benchmark_texts) / elapsed_seconds, 2))
print("embedding norm range:", embedding_norms.min(), "to", embedding_norms.max())
print(
    "peak GPU memory (MiB):",
    round(torch.cuda.max_memory_allocated() / 1024**2, 1),
)

CUDA available: True
model device: cuda:0


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

sample size: 500
embedding shape: (500, 768)
ID count: 500
elapsed seconds: 3.57
texts per second: 140.12
embedding norm range: 0.99999994 to 1.0000001
peak GPU memory (MiB): 338.5


## Menguji implementasi `encode_semantic_texts()`

Bagian ini mengulang benchmark 500 teks melalui fungsi reusable. Hasilnya harus sama dengan benchmark langsung: bentuk `(500, 768)`, tipe `float32`, dan norm mendekati 1. Jika berbeda, ada masalah pada implementasi fungsi.


Bagian ini mengulang benchmark 500 teks, tetapi melalui `encode_semantic_texts()`. Tujuannya memastikan fungsi reusable menghasilkan output yang sama dengan pemanggilan `model.encode` langsung.

In [21]:
from movie_recommender.features.sbert import encode_semantic_texts

benchmark_texts = texts.sample(n=500, random_state=42)

benchmark_embeddings = encode_semantic_texts(
    model=model,
    texts=benchmark_texts,
    batch_size=8,
)

norms = np.linalg.norm(benchmark_embeddings, axis=1)

print("shape:", benchmark_embeddings.shape)
print("dtype:", benchmark_embeddings.dtype)
print("rows:", len(benchmark_texts))
print("norm min:", norms.min())
print("norm max:", norms.max())

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

shape: (500, 768)
dtype: float32
rows: 500
norm min: 0.99999994
norm max: 1.0000001


## Menyimpan hasil preprocessing 

In [22]:
from movie_recommender.utils.paths import RESULTS_DIR, SEMANTIC_ARTIFACTS_DIR

SEMANTIC_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

semantic_text_frame = pd.DataFrame(
    {
        "id": movies_dedup["id"],
        "semantic_text": semantic_text,
    }
)
semantic_text_frame.to_parquet(
    SEMANTIC_ARTIFACTS_DIR / "semantic_text.parquet",
    index=False,
    compression="zstd",
)

token_length_frame = pd.DataFrame(
    {
        "id": movies_dedup.loc[texts.index, "id"],
        "token_length": token_lengths,
    }
).reset_index(drop=True)
token_length_frame.to_parquet(
    SEMANTIC_ARTIFACTS_DIR / "token_lengths.parquet",
    index=False,
    compression="zstd",
)

print("semantic_text rows:", len(semantic_text_frame))
print("token_length rows:", len(token_length_frame))

semantic_text rows: 429755
token_length rows: 396254


In [23]:
loaded_text = pd.read_parquet(
    SEMANTIC_ARTIFACTS_DIR / "semantic_text.parquet"
)
loaded_lengths = pd.read_parquet(
    SEMANTIC_ARTIFACTS_DIR / "token_lengths.parquet"
)

print(loaded_text.shape, loaded_lengths.shape)
print(loaded_text.head())
print(loaded_lengths.head())

(429755, 2) (396254, 2)
       id                                      semantic_text
0   27205  Overview: Cobb, a skilled thief who commits co...
1  157336  Overview: The adventures of a group of explore...
2     155  Overview: Batman raises the stakes in his war ...
3   19995  Overview: In the 22nd century, a paraplegic Ma...
4   24428  Overview: When an unexpected enemy emerges and...
       id  token_length
0   27205           123
1  157336           153
2     155           147
3   19995            97
4   24428           117


## Full Encoding

Tahap ini mengubah seluruh 396.254 `semantic_text` menjadi embedding dan menyimpan tiga artifact: matriks embedding, mapping movie ID, dan konfigurasi eksperimen.

Encoding dijalankan sekali dengan model pretrained. Bobot model tidak diubah; tidak ada training atau fine-tuning pada tahap ini.


In [24]:
print(len(texts))
print(movies_dedup.shape)
print(model.device)
print(max_seq_length)

396254
(429755, 29)
cuda:0
512


In [25]:
import json
import time
from datetime import datetime, timezone

import numpy as np
import torch
import sentence_transformers

from movie_recommender.utils.paths import SEMANTIC_ARTIFACTS_DIR

SEMANTIC_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 8

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
start_time = time.perf_counter()

semantic_embeddings = encode_semantic_texts(
    model=model,
    texts=texts,
    batch_size=BATCH_SIZE,
)

torch.cuda.synchronize()
elapsed_seconds = time.perf_counter() - start_time

semantic_movie_ids = movies_dedup.loc[texts.index, "id"].to_numpy(
    dtype=np.int64
)

np.save(
    SEMANTIC_ARTIFACTS_DIR / "embeddings.npy",
    semantic_embeddings,
)
np.save(
    SEMANTIC_ARTIFACTS_DIR / "movie_ids.npy",
    semantic_movie_ids,
)

config = {
    "model_name": MODEL_NAME,
    "embedding_dim": int(semantic_embeddings.shape[1]),
    "dtype": str(semantic_embeddings.dtype),
    "n_texts": int(semantic_embeddings.shape[0]),
    "n_movies": int(len(movies_dedup)),
    "n_empty_semantic_text": int(semantic_text.eq("").sum()),
    "max_seq_length": int(max_seq_length),
    "batch_size": BATCH_SIZE,
    "device": str(model.device),
    "truncation": "model default (max_seq_length)",
    "n_over_250_tokens": int(token_lengths.gt(250).sum()),
    "n_over_512_tokens": int(token_lengths.gt(max_seq_length).sum()),
    "elapsed_seconds": round(elapsed_seconds, 2),
    "peak_gpu_memory_mib": round(
        torch.cuda.max_memory_allocated() / 1024**2, 1
    ),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "sentence_transformers_version": sentence_transformers.__version__,
    "torch_version": torch.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
}

(SEMANTIC_ARTIFACTS_DIR / "config.json").write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)

print(json.dumps(config, indent=2))

Batches:   0%|          | 0/49532 [00:00<?, ?it/s]

{
  "model_name": "sentence-transformers/multi-qa-distilbert-cos-v1",
  "embedding_dim": 768,
  "dtype": "float32",
  "n_texts": 396254,
  "n_movies": 429755,
  "n_empty_semantic_text": 33501,
  "max_seq_length": 512,
  "batch_size": 8,
  "device": "cuda:0",
  "truncation": "model default (max_seq_length)",
  "n_over_250_tokens": 520,
  "n_over_512_tokens": 16,
  "elapsed_seconds": 1671.52,
  "peak_gpu_memory_mib": 393.8,
  "created_at_utc": "2026-11-12T03:55:47.407023+00:00",
  "sentence_transformers_version": "6.2.0",
  "torch_version": "2.15.0+cu134",
  "numpy_version": "2.4.6",
  "pandas_version": "3.1.1"
}


### Verifikasi Artifact

File `.npy` dibaca kembali dengan `mmap_mode="r"` agar matriks 1,2 GB tidak langsung dimuat ke RAM. Pemeriksaan mencakup bentuk, tipe data, jumlah ID unik, norm embedding, dan nilai non-finite.


In [26]:
import numpy as np

from movie_recommender.utils.paths import SEMANTIC_ARTIFACTS_DIR

embeddings = np.load(
    SEMANTIC_ARTIFACTS_DIR / "embeddings.npy",
    mmap_mode="r",
)
movie_ids = np.load(SEMANTIC_ARTIFACTS_DIR / "movie_ids.npy")

print("embeddings:", embeddings.shape, embeddings.dtype)
print("movie_ids:", movie_ids.shape, movie_ids.dtype)
print("unique ids:", len(np.unique(movie_ids)))

sample_norms = np.linalg.norm(embeddings[:1000], axis=1)
print("norm min:", sample_norms.min())
print("norm max:", sample_norms.max())
print("finite sample:", np.isfinite(embeddings[:1000]).all())

embeddings: (396254, 768) float32
movie_ids: (396254,) int64
unique ids: 396254
norm min: 0.99999994
norm max: 1.0000001
finite sample: True


### Spot Check Alignment

Beberapa movie ID yang sudah dikenal diperiksa posisinya. Tujuannya memastikan baris embedding benar-benar milik film yang dimaksud, bukan hanya jumlah barisnya yang sama.


In [27]:
for movie_id in [27205, 157336, 155, 862]:
    rows = np.where(movie_ids == movie_id)[0]

    title = movies_dedup.loc[
        movies_dedup["id"] == movie_id, "title"
    ].item()

    print(movie_id, title, "rows:", rows)

27205 Inception rows: [0]
157336 Interstellar rows: [1]
155 The Dark Knight rows: [2]
862 Toy Story rows: [73]


## Analisis Hasil Rekomendasi Semantik

Bagian ini memakai embedding yang sudah tersimpan untuk mengambil Top-K film mirip. Tujuannya bukan evaluasi formal, melainkan sanity check: memastikan hasilnya masuk akal secara kualitatif dan mapping ID-nya benar.

Semua perhitungan dilakukan pada embedding yang sudah L2-normalized, sehingga cosine similarity setara dengan dot product.


In [31]:
movie_index = movies_dedup.set_index("id")


def movie_year(value):
    date = pd.to_datetime(value, errors="coerce")
    return None if pd.isna(date) else int(date.year)


def movie_profile(movie_id: int) -> dict:
    row = movie_index.loc[movie_id]
    return {
        "id": movie_id,
        "title": row["title"],
        "year": movie_year(row["release_date"]),
        "genres": row["genres"],
        "directors": row["directors"],
        "overview": row["overview"],
    }

### Genre Overlap

Genre overlap menghitung Jaccard similarity antara himpunan genre query dan genre rekomendasi. Nilai ini hanya alat bantu interpretasi, bukan ground truth dan tidak dipakai untuk ranking.


In [32]:
def genre_overlap(genres_a, genres_b) -> float:
    if pd.isna(genres_a) or pd.isna(genres_b):
        return 0.0

    set_a = {genre.strip() for genre in str(genres_a).split(",")}
    set_b = {genre.strip() for genre in str(genres_b).split(",")}

    union = set_a | set_b
    if not union:
        return 0.0

    return len(set_a & set_b) / len(union)

### Mengambil Top-K dengan Metadata

Fungsi ini mengambil skor kemiripan untuk satu film, mengeluarkan film itu sendiri, lalu mengambil K baris teratas. Hasilnya digabungkan dengan judul, tahun, genre, sutradara, rating, dan genre overlap agar mudah diperiksa.


In [33]:
def top_k_similar(movie_id: int, k: int = 10) -> pd.DataFrame:
    row = int(np.where(movie_ids == movie_id)[0][0])

    scores = embeddings @ embeddings[row]
    scores[row] = -np.inf

    top_rows = np.argsort(scores)[::-1][:k]
    top_ids = movie_ids[top_rows]

    result = movie_index.loc[
        top_ids,
        ["title", "release_date", "genres", "directors", "vote_average"],
    ].copy()

    result.insert(0, "rank", range(1, len(result) + 1))
    result.insert(1, "recommended_movie_id", top_ids)
    result["similarity"] = scores[top_rows]
    result["year"] = result["release_date"].apply(movie_year)
    result["genre_overlap"] = [
        genre_overlap(movie_index.loc[movie_id, "genres"], genres)
        for genres in result["genres"]
    ]

    return result.drop(columns=["release_date"]).reset_index(drop=True)

### Menampilkan Query Secara Deskriptif

Fungsi ini menampilkan profil film query dan tabel Top-K. Profil membantu memahami konteks: genre, sutradara, dan ringkasan sinopsis. Tabel menambahkan tahun, genre, similarity, dan genre overlap pada setiap rekomendasi.


In [35]:
def show_query(movie_id: int, k: int = 10) -> None:
    profile = movie_profile(movie_id)
    table = top_k_similar(movie_id, k)

    print("=" * 90)
    print(f"QUERY     : {profile['title']} ({profile['year']}) | id={profile['id']}")
    print(f"Genres    : {profile['genres']}")
    print(f"Directors : {profile['directors']}")
    print(f"Overview  : {str(profile['overview'])[:300]}...")
    print("-" * 90)
    print(
        table[
            ["rank", "title", "year", "genres", "similarity", "genre_overlap"]
        ].to_string(index=False)
    )
    print()

### Menjalankan Contoh Query

Empat film dipakai sebagai contoh: satu film science fiction, satu space epic, satu superhero, dan satu animasi. Kombinasi ini membantu melihat apakah embedding menangkap tema yang berbeda.


In [37]:
for query_id in [27205, 157336, 155, 862]:
    show_query(query_id, k=10)

QUERY     : Inception (2010) | id=27205
Genres    : Action, Science Fiction, Adventure
Directors : Christopher Nolan
Overview  : Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as payment for a task considered to be impossible: "inception", the implantation of another person's idea into a target's subconscious....
------------------------------------------------------------------------------------------
 rank                        title  year                                  genres  similarity  genre_overlap
    1              Respect the Jux  2022          Crime, Drama, Thriller, Action    0.560876       0.166667
    2              American Psycho  2000                  Thriller, Drama, Crime    0.549396       0.000000
    3                      Vivegam  2017                        Action, Thriller    0.543944       0.250000
    4          The Hunter's Prayer  2017                        

### Interpretasi Sanity Check

Hasil yang diamati pada empat query:

- **Interstellar** menghasilkan cluster film luar angkasa dan eksplorasi. Ini indikasi kuat bahwa embedding menangkap tema, bukan hanya kata yang sama.
- **The Dark Knight** menghasilkan film Batman dan DC. Franchise mendominasi, yang wajar karena kemiripan teks dan karakter sangat tinggi.
- **Inception** menghasilkan film crime, heist, dan psychological thriller. Relevan secara tema, tetapi tidak semua orang akan menganggapnya mirip.
- **Toy Story** menghasilkan tiga sekuel resmi, lalu beberapa kartun lama dengan karakter bernama Woody. Ini menunjukkan embedding juga dapat tertarik pada overlap leksikal, bukan hanya kesamaan tema.

Skor similarity berada di kisaran 0,52–0,72. Nilai ini normal untuk embedding teks dan penting dicatat karena distribusi skor akan dibandingkan dengan skor metadata pada tahap fusion.

Semua contoh di atas adalah film berbahasa Inggris. Model yang dipakai berfokus pada bahasa Inggris, sehingga kualitas untuk film non-Inggris belum terbukti dan perlu diuji terpisah.


### Menyimpan Hasil Deskriptif

Hasil Top-K disimpan sebagai CSV agar bisa diperiksa ulang tanpa menjalankan ulang model. Kolom query dan metadata rekomendasi ikut disimpan untuk keperluan dokumentasi.


In [42]:
recommendation_dir = RESULTS_DIR / "recommendations"
recommendation_dir.mkdir(parents=True, exist_ok=True)

descriptive_path = (
    recommendation_dir / "semantic_top10_descriptive.csv"
)

query_ids = [27205, 157336, 155, 862]
rows = []

for query_id in query_ids:
    profile = movie_profile(query_id)
    table = top_k_similar(query_id, 10)

    table.insert(0, "query_movie_id", query_id)
    table.insert(1, "query_title", profile["title"])
    table.insert(2, "query_year", profile["year"])

    rows.append(table)

pd.concat(rows, ignore_index=True).to_csv(
    descriptive_path,
    index=False,
)


### Membaca Kembali Hasil

CSV dibaca kembali untuk memastikan file tersimpan dengan kolom yang benar dan bisa dipakai di analisis berikutnya.


In [43]:
df = pd.read_csv(descriptive_path)
df


,query_movie_id,query_title,query_year,rank,recommended_movie_id,title,genres,directors,vote_average,similarity,year,genre_overlap
0,27205,Inception,2010,1,962675,Respect the Jux,"Crime, Drama, Thriller, Action",G.H. Goba,3.500,0.560876,2022.0,0.166667
1,27205,Inception,2010,2,1359,American Psycho,"Thriller, Drama, Crime",Mary Harron,7.400,0.549396,2000.0,0.000000
2,27205,Inception,2010,3,438731,Vivegam,"Action, Thriller",Siva,4.900,0.543944,2017.0,0.250000
3,27205,Inception,2010,4,302150,The Hunter's Prayer,"Action, Thriller",Jonathan Mostow,5.715,0.537787,2017.0,0.250000
4,27205,Inception,2010,5,515853,Gultoo,"Crime, Drama",Janardhan Chikkanna,7.714,0.536684,2018.0,0.000000
5,27205,Inception,2010,6,119283,Parker,"Action, Crime",Taylor Hackford,6.018,0.525582,2013.0,0.250000
6,27205,Inception,2010,7,607231,ECCO,"Action, Crime, Drama",Ben Medina,3.800,0.519688,2019.0,0.200000
7,27205,Inception,2010,8,649409,No Sudden Move,"Crime, Drama, Mystery",Steven Soderbergh,6.400,0.516299,2021.0,0.000000
8,27205,Inception,2010,9,610236,Making a Deal with the Devil,Drama,Frank D'Angelo,0.000,0.516269,2019.0,0.000000
9,27205,Inception,2010,10,281,Strange Days,"Crime, Drama, Science Fiction, Thriller",Kathryn Bigelow,6.989,0.515185,1995.0,0.166667


## Uji Film Non-Inggris

Dua film dengan teks berbahasa Rusia dipakai untuk melihat apakah model menghasilkan tetangga yang relevan atau justru mengelompok berdasarkan bahasa/script. Ini bukan evaluasi formal, melainkan pemeriksaan keterbatasan model.

In [44]:
for query_id in [77994, 64060]:
    show_query(query_id, k=10)

QUERY     : Yegor's Grief (2008) | id=77994
Genres    : Comedy
Directors : Mariya Makhanko
Overview  : У Егора проблемы. Он занял денег для продюсирования молодых дарований, а молодые дарования, попользовавшись им – ушли к другому музыкальному продюсеру. Теперь Егор должен до конца недели вернуть деньги гламурному бандиту Максимилиану, или представить ему новую талантливую звезду, иначе ему конец. По...
------------------------------------------------------------------------------------------
 rank                   title  year                   genres  similarity  genre_overlap
    1        Padeniye Kondora  1982                    Drama    0.827913       0.000000
    2          Night Accident  1980                  Mystery    0.816560       0.000000
    3           Princess Mary  1955           Drama, Romance    0.814257       0.000000
    4            Shizofreniya  1997         Action, Thriller    0.808558       0.000000
    5      Life by somersault  2003                   Comedy  

In [45]:
non_english_rows = []

for query_id in [77994, 64060]:
    profile = movie_profile(query_id)
    table = top_k_similar(query_id, 10)

    table.insert(0, "query_movie_id", query_id)
    table.insert(1, "query_title", profile["title"])
    table.insert(2, "query_year", profile["year"])

    non_english_rows.append(table)

pd.concat(non_english_rows, ignore_index=True).to_csv(
    recommendation_dir / "semantic_top10_non_english.csv",
    index=False,
)

In [46]:
summary = {
    "n_movies": len(movies_dedup),
    "n_semantic_texts": len(texts),
    "n_empty_semantic_text": int(semantic_text.eq("").sum()),
    "max_seq_length": int(max_seq_length),
    "n_over_250": int(token_lengths.gt(250).sum()),
    "n_over_512": int(token_lengths.gt(max_seq_length).sum()),
    "token_p50": float(token_lengths.quantile(0.50)),
    "token_p90": float(token_lengths.quantile(0.90)),
    "token_p99": float(token_lengths.quantile(0.99)),
    "token_max": int(token_lengths.max()),
}

audit_path = RESULTS_DIR / "semantic_text_audit.csv"
pd.Series(summary, name="value").to_csv(audit_path)

print(audit_path)
print(pd.Series(summary))

D:\College Life\Skripsiw\hybrid-movie-recommender\results\semantic_text_audit.csv
n_movies                 429755.0
n_semantic_texts         396254.0
n_empty_semantic_text     33501.0
max_seq_length              512.0
n_over_250                  520.0
n_over_512                   16.0
token_p50                    61.0
token_p90                   140.0
token_p99                   215.0
token_max                   782.0
dtype: float64


### Cek Komposisi Bahasa

Fungsi `language_mix()` menghitung distribusi `original_language` pada Top-K hasil rekomendasi. Analisis ini bersifat deskriptif dan tidak dipakai untuk ranking.

Perlu diperhatikan bahwa `original_language` dan `spoken_languages` dapat berbeda atau tidak lengkap. Karena itu, hasilnya dibaca sebagai indikasi awal, bukan klasifikasi bahasa yang sempurna.

In [47]:
def language_mix(movie_id: int, k: int = 10) -> pd.Series:
    table = top_k_similar(movie_id, k)
    languages = movie_index.loc[
        table["recommended_movie_id"], "original_language"
    ]
    return languages.value_counts()


for query_id in [27205, 157336, 155, 862, 77994, 64060]:
    print("\nQUERY:", query_id)
    print(language_mix(query_id))


QUERY: 27205
original_language
en    8
ta    1
kn    1
Name: count, dtype: int64

QUERY: 157336
original_language
en    9
xx    1
Name: count, dtype: int64

QUERY: 155
original_language
en    10
Name: count, dtype: int64

QUERY: 862
original_language
en    10
Name: count, dtype: int64

QUERY: 77994
original_language
ru    7
en    3
Name: count, dtype: int64

QUERY: 64060
original_language
ru    6
en    4
Name: count, dtype: int64


In [48]:
summary_int = {
    "n_movies": len(movies_dedup),
    "n_semantic_texts": len(texts),
    "n_empty_semantic_text": int(semantic_text.eq("").sum()),
    "max_seq_length": int(max_seq_length),
    "n_over_250": int(token_lengths.gt(250).sum()),
    "n_over_512": int(token_lengths.gt(max_seq_length).sum()),
    "token_p50": int(token_lengths.quantile(0.50)),
    "token_p90": int(token_lengths.quantile(0.90)),
    "token_p99": int(token_lengths.quantile(0.99)),
    "token_max": int(token_lengths.max()),
}

pd.Series(summary_int, name="value").to_csv(
    RESULTS_DIR / "semantic_text_audit.csv"
)

### Temuan Awal: Efek Bahasa

Dua film dengan teks berbahasa Rusia menghasilkan Top-10 yang didominasi film berbahasa Rusia. Berdasarkan `original_language`, komposisinya 7 film `ru` dan 3 film `en` untuk Yegor's Grief, serta 6 film `ru` dan 4 film `en` untuk Sokrovishche.

Similarity untuk kedua query berada di kisaran 0,77–0,83, lebih tinggi daripada contoh film berbahasa Inggris (0,52–0,72). Namun nilai ini tidak dapat langsung diartikan sebagai kualitas rekomendasi yang lebih baik. Model yang digunakan berfokus pada bahasa Inggris, sehingga kemiripan tinggi pada teks Cyrillic dapat dipengaruhi oleh kesamaan bahasa/script, bukan hanya kesamaan tema.

Sebagian metadata bahasa juga tidak konsisten. Beberapa film berlabel `original_language = en` memiliki `spoken_languages = Russian` atau kosong. Karena itu, analisis bahasa perlu mempertimbangkan kedua kolom dan tidak boleh hanya mengandalkan `original_language`.

Temuan ini dicatat sebagai keterbatasan dan perlu diuji lebih lanjut sebelum menarik kesimpulan tentang kualitas lintas bahasa.